Verify GPU

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-40GB


Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/sqlcoder_ft'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

Mounted at /content/drive
Output dir: /content/drive/MyDrive/Colab Notebooks/sqlcoder_ft


Install dependencies

In [ ]:
!pip install -q -U transformers peft trl accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 150.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.9 MB/s eta 0:00:00


RESTART RUNTIME HERE, Re-mount Drive after restart, verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

import os
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/sqlcoder_ft'

Mounted at /content/drive
True
NVIDIA A100-SXM4-40GB


Load schema and training pairs

In [ ]:
import importlib.util, json

spec = importlib.util.spec_from_file_location(
    "schema_prompt",
    "/content/drive/MyDrive/Colab Notebooks/schema_prompt.py"
)
schema_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(schema_mod)
SCHEMA_CONTEXT = schema_mod.SCHEMA_CONTEXT

print(SCHEMA_CONTEXT[:200])

with open('/content/drive/MyDrive/Colab Notebooks/training_pairs.json') as f:
    pairs = json.load(f)

print(f"Loaded {len(pairs)} training pairs")


### PostgreSQL schema: bl_dm
### Retail sales warehouse. Always start from the fact table.

### FACT TABLE
Table: bl_dm.fct_transactions_dd
Columns:
  transaction_src_id        BIGINT
  product_surr_
Loaded 300 training pairs


Load tokenizer FIRST

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "defog/sqlcoder-7b-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"EOS token: {tokenizer.eos_token!r}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

EOS token: '</s>'


Build training dataset

In [ ]:
from datasets import Dataset

TEMPLATE = """### Instructions:
Your task is to convert a question into a SQL query, given a Postgres database schema.
Adhere to these rules:
- Output ONLY the SQL query. No explanation, no preamble, no markdown.
- Always qualify table names with the bl_dm schema.
- Always use table aliases and qualify all column references.
- Carefully read the schema notes before writing any query.

### Input:
{schema}

Question: {question}

### Response:
{sql}"""

# Build the dataset — single source of truth, no double EOS
formatted = []
for p in pairs:
    text = TEMPLATE.format(
        schema=SCHEMA_CONTEXT,
        question=p['question'],
        sql=p['sql']
    ) + tokenizer.eos_token
    formatted.append(text)   # plain string, not dict

print(f"Total training examples: {len(formatted)}")

# Token length audit (now works because formatted is list of strings)
lengths = [len(tokenizer.encode(t)) for t in formatted]
print(f"Token length — max: {max(lengths)}, avg: {sum(lengths)//len(lengths)}, min: {min(lengths)}")
print(f"Examples exceeding 3056 tokens: {sum(1 for l in lengths if l > 3056)}")
print(f"Examples exceeding 2800 tokens: {sum(1 for l in lengths if l > 2800)}")


Total training examples: 300
Token length — max: 2626, avg: 2490, min: 2427
Examples exceeding 3056 tokens: 0
Examples exceeding 2800 tokens: 0


Build HF dataset and split train/evaluation

In [ ]:


ds = Dataset.from_dict({"text": formatted})
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]
print(f"Train: {len(train_ds)}  Eval: {len(eval_ds)}")



Train: 270  Eval: 30


Load model with 4-bit quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print(f"Model loaded — params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded — params: 3.50B


Attach LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    r=32,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 79,953,920 || all params: 6,818,500,608 || trainable%: 1.1726


Build trainer with completion-only loss

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        bf16=True,
        fp16=False,
        logging_steps=5,
        save_steps=100,
        save_total_limit=2,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        report_to="none",
        max_length=3056,
        packing=False,
        eval_strategy="steps",
        eval_steps=10,
        completion_only_loss=True,
    )
)
print("Trainer ready")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Trainer ready


Train

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,0.911857,0.834798,0.682902,199049.000000,0.808181
20,0.566295,0.409724,0.509343,398170.000000,0.895062
30,0.099057,0.043715,0.066865,597727.000000,0.990682
40,0.023014,0.022200,0.026817,792091.000000,0.995276
50,0.018961,0.017609,0.022141,991277.000000,0.996477
60,0.015893,0.014910,0.016259,1190237.000000,0.997005
70,0.013615,0.014340,0.014363,1384281.000000,0.997072
80,0.012874,0.014140,0.013866,1583382.000000,0.997072
90,0.012924,0.014113,0.013788,1782685.000000,0.997097
100,0.014033,0.014112,0.013773,1981958.000000,0.997072


TrainOutput(global_step=102, training_loss=0.18993476126343012, metrics={'train_runtime': 1143.1559, 'train_samples_per_second': 0.709, 'train_steps_per_second': 0.089, 'total_flos': 8.142945218032435e+16, 'train_loss': 0.18993476126343012, 'epoch': 3.0})

Save the LoRA adapter

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

Adapter saved to /content/drive/MyDrive/Colab Notebooks/sqlcoder_ft/lora_adapter


Merge adapter into base model

In [ ]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.2 MB/s eta 0:00:00


In [ ]:
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"
LOCAL_MERGED = "/content/merged_model"


for var_name in ['trainer', 'model', 'base_model', 'merged_model']:
    if var_name in dir():
        try:
            exec(f"del {var_name}")
        except:
            pass

gc.collect()
torch.cuda.empty_cache()

print("Loading base model in fp16...")
base_model = AutoModelForCausalLM.from_pretrained(
    "defog/sqlcoder-7b-2",
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Loading PEFT adapter...")
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Merging LoRA into base model...")
merged_model = merged_model.merge_and_unload()

print("Saving merged model locally...")
merged_model.save_pretrained(LOCAL_MERGED, safe_serialization=True, max_shard_size="2GB")

print("Saving tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_MERGED)
print("Merge done")

Loading base model in fp16...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading PEFT adapter...
Merging LoRA into base model...
Saving merged model locally...


Writing model shards:   0%|          | 0/7 [00:00<?, ?it/s]

Saving tokenizer...
Merge done


Build llama.cpp for GGUF conversion

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt
!apt-get install -y cmake
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j4 2>&1 | tail -5

!ls /content/llama.cpp/convert*.py

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 97995, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 97995 (delta 87), reused 57 (delta 57), pack-reused 97850 (from 2)
Receiving objects: 100% (97995/97995), 395.87 MiB | 19.00 MiB/s, done.
Resolving deltas: 100% (69975/69975), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 66.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 151.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

 Copy original tokenizer.model

In [ ]:
from transformers import AutoTokenizer
import shutil, glob

tokenizer = AutoTokenizer.from_pretrained("defog/sqlcoder-7b-2", use_fast=False)
tokenizer.save_pretrained(LOCAL_MERGED)

for f in glob.glob("/root/.cache/huggingface/hub/**/tokenizer.model", recursive=True):
    shutil.copy(f, LOCAL_MERGED)
    print(f"Copied {f}")


Copied /root/.cache/huggingface/hub/models--defog--sqlcoder-7b-2/snapshots/7e5b6f7981c0aa7d143f6bec6fa26625bdfcbe66/tokenizer.model


Convert to GGUF (f16) then quantize to q4_k_m

In [ ]:
LOCAL_F16  = "/content/sqlcoder_ft_f16.gguf"
LOCAL_Q4   = "/content/sqlcoder_ft_q4km.gguf"
FINAL_GGUF = f"{OUTPUT_DIR}/sqlcoder_ft_q4km.gguf"

print("Converting to f16 GGUF...")
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model --outfile {LOCAL_F16} --outtype f16

print("Quantizing to q4_k_m...")
!/content/llama.cpp/build/bin/llama-quantize {LOCAL_F16} {LOCAL_Q4} q4_k_m

print("Copying final GGUF to Drive...")
import shutil
shutil.copy(LOCAL_Q4, FINAL_GGUF)
print(f"Done: {FINAL_GGUF}")
print(f"Size: {os.path.getsize(FINAL_GGUF) / 1e9:.2f} GB")

Converting to f16 GGUF...
INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00005-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00006-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00007-of-00007.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {4096, 32016}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --

sanity-check the merged model before download

In [ ]:
test_prompt = """### Instructions:
Your task is to convert a question into a SQL query, given a Postgres database schema.
Adhere to these rules:
- Output ONLY the SQL query. No explanation, no preamble, no markdown.
- Always qualify table names with the bl_dm schema.
- Always use table aliases and qualify all column references.
- Carefully read the schema notes before writing any query.

### Input:
""" + SCHEMA_CONTEXT + """

Question: What is the total revenue broken down by year?

### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


SELECT d.year, SUM(f.transaction_sale_amount) AS total_revenue FROM bl_dm.fct_transactions_dd f JOIN bl_dm.dim_dates d ON f.event_date_key = d.date_key GROUP BY d.year ORDER BY d.year
